# Google Data Center Power Trace Analysis

# Modified/updated by Ben Logan

This colab demonstrates querying the Google data center power traces with bigquery, visualizing them with [Altair](https://altair-viz.github.io/), and analyzing them in conjunction with the 2019 Google cluster data.

**Important:** in order to be able to run the queries you will need to:

1. Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.
1. [Enable billing](https://support.google.com/cloud/answer/6293499#enable-billing) for the project.
1. [Enable BigQuery](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) APIs for the project.

To begin with, we'll authenticate with GCP and import the python libraries necessary to execute this colab.

In [ ]:
#@title Please input your project id
import altair as alt
import numpy as np
import pandas as pd
from google.cloud import bigquery
# Provide credentials to the runtime
from google.colab import auth
from google.cloud.bigquery import magics

auth.authenticate_user()
print('Authenticated')
project_id = 'black-backup-489114-h7' #@param {type: "string"}
# Set the default project id for %bigquery magic
magics.context.project = project_id

# Use the client to run queries constructed from a more complicated function.
client = bigquery.Client(project=project_id)


Authenticated


## Basic Queries

Here are some examples of using the [bigquery magic](https://cloud.google.com/python/docs/reference/bigquery/latest/index.html) to query the power traces.

First we'll calculate the average production utilization for a single power domain.


In [ ]:
%%bigquery
SELECT
  AVG(production_power_util) AS average_production_power_util
FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu10

Query is running:   0%|          |

Downloading:   0%|          |

,average_production_power_util
0,0.601692



Now let's find the minimum and maximum measured power utilization for each cell. We use bigquery [wildcard tables](https://cloud.google.com/bigquery/docs/querying-wildcard-tables) in order to conveniently query all trace tables at once.

In [ ]:
%%bigquery
SELECT
  cell,
  MIN(measured_power_util) AS minimum_measured_power_util,
  MAX(measured_power_util) AS maximum_measured_power_util
FROM `google.com:google-cluster-data.powerdata_2019.cell*`
GROUP BY cell

Query is running:   0%|          |

Downloading:   0%|          |

,cell,minimum_measured_power_util,maximum_measured_power_util
0,d,0.562,0.767
1,g,0.554,0.831
2,a,0.611,0.791
3,f,0.544,0.794
4,h,0.373,0.718
5,e,0.567,0.845
6,b,0.672,0.799
7,c,0.585,0.829
8,i,0.595,0.774
9,j,0.549,0.669


Modifying the previous query to also group by `pdu` gives us the maximum and minimum measured power utilization per power domain.

In [ ]:
%%bigquery
SELECT
  cell,
  pdu,
  MIN(measured_power_util) AS minimum_measured_power_util,
  MAX(measured_power_util) AS maximum_measured_power_util
FROM `google.com:google-cluster-data.powerdata_2019.cell*`
GROUP BY cell, pdu
ORDER BY maximum_measured_power_util

Query is running:   0%|          |

Downloading:   0%|          |

,cell,pdu,minimum_measured_power_util,maximum_measured_power_util
0,h,pdu45,0.373,0.622
1,h,pdu49,0.500,0.643
2,j,mvpp2,0.549,0.669
3,h,pdu54,0.522,0.677
4,h,pdu47,0.496,0.691
5,h,pdu44,0.493,0.693
6,h,pdu55,0.529,0.694
7,h,pdu50,0.518,0.696
8,h,pdu53,0.524,0.698
9,h,pdu51,0.532,0.706


## Measured and Production Power over Time

We provide traces for two clusters of Google's new Medium Voltage Power Plane (MVPP) data center design, described in [the paper](https://research.google/pubs/pub49032/). Let's plot the measured and estimated production power utilization of one of these MVPPs: mvpp1. We'll limit this visualization to the first 15 days of the trace period (the first 4320 datapoints of the trace).

In [ ]:
%%bigquery mvpp1_df
SELECT
  time,
  measured_power_util,
  production_power_util
FROM `google.com:google-cluster-data`.powerdata_2019.celli_mvpp1
ORDER BY time
LIMIT 4320

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
alt.Chart(mvpp1_df).mark_line().transform_fold(
    ["measured_power_util", "production_power_util"]).encode(
    x="time:Q",
    y=alt.X("value:Q", scale=alt.Scale(zero=False)),
    color="key:N"
).properties(width=700, height=75)

alt.Chart(...)

## CPU and Power over Time

As an example of joining the power traces with the cluster traces, we'll plot the average CPU utilization and power utilization per hour.

You may remember this query from the [cluster analysis colab](https://github.com/google/cluster-data/blob/master/clusterdata_analysis_colab.ipynb). It's been modified slightly--see `query_per_tier_utilization_time_series` in particular.

In [ ]:
def machines_in_pdu(pdu_number):
  return '''
SELECT machine_id
FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
WHERE pdu = 'pdu{pdu_number}'
  '''.format(pdu_number=pdu_number)

def query_cell_capacity(cell, pdu_number):
  return '''
SELECT SUM(cpu_cap) AS cpu_capacity
FROM (
  SELECT machine_id, MAX(capacity.cpus) AS cpu_cap,
  FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
  WHERE machine_id IN ({machine_query})
  GROUP BY 1
)
  '''.format(cell=cell, machine_query=machines_in_pdu(pdu_number))

def query_per_instance_usage_priority(cell, pdu_num):
  return '''
SELECT u.time AS time,
  u.collection_id AS collection_id,
  u.instance_index AS instance_index,
  e.priority AS priority,
  CASE
    WHEN e.priority BETWEEN 0 AND 99 THEN '1_free'
    WHEN e.priority BETWEEN 100 AND 115 THEN '2_beb'
    WHEN e.priority BETWEEN 116 AND 119 THEN '3_mid'
    ELSE '4_prod'
  END AS tier,
  u.cpu_usage AS cpu_usage
FROM (
  SELECT start_time AS time,
    collection_id,
    instance_index,
    machine_id,
    average_usage.cpus AS cpu_usage
  FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
  WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
    AND (end_time - start_time) >= (5 * 60 * 1e6)
) AS u JOIN (
  SELECT collection_id, instance_index, machine_id,
    MAX(priority) AS priority
  FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_events
  WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
    AND machine_id IN ({machine_query})
  GROUP BY 1, 2, 3
) AS e ON u.collection_id = e.collection_id
  AND u.instance_index = e.instance_index
  AND u.machine_id = e.machine_id
  '''.format(cell=cell, machine_query=machines_in_pdu(pdu_num))

def query_per_tier_utilization_time_series(cell, pdu_num, cpu_capacity):
  return '''
SELECT * FROM (
  SELECT CAST(FLOOR(time/(1e6 * 60 * 60)) AS INT64) AS hour_index,
    tier,
    SUM(cpu_usage) / (12 * {cpu_capacity}) AS avg_cpu_usage,
  FROM ({table})
  GROUP BY 1, 2)
JOIN (
  SELECT CAST(FLOOR(time/(1e6 * 60 * 60)) AS INT64) AS hour_index,
  pdu,
  AVG(measured_power_util) as avg_measured_power_util,
  AVG(production_power_util) AS avg_production_power_util
  FROM `google.com:google-cluster-data`.`powerdata_2019.cell{cell}_pdu{pdu_num}`
  GROUP BY hour_index, pdu
) USING (hour_index)
  '''.format(table=query_per_instance_usage_priority(cell, pdu_num),
             cpu_capacity=cpu_capacity, cell=cell, pdu_num=pdu_num)

def run_query_utilization_per_time_time_series(cell, pdu_num):
  cell_cap = client.query(query_cell_capacity(cell, pdu_num)).to_dataframe()
  query = query_per_tier_utilization_time_series(
      cell,
      pdu_num,
      cell_cap['cpu_capacity'][0])
  time_series = client.query(query).to_dataframe()
  return time_series

CELL='f'
PDU_NUM='17'
hourly_usage = run_query_utilization_per_time_time_series(CELL, PDU_NUM)

Plot power utilization on top of the CPU utilization graph.

In [ ]:
# CPU graph
cpu = alt.Chart().mark_area().encode(
        alt.X('hour_index:N'),
        alt.Y('avg_cpu_usage:Q'),
        color=alt.Color('tier', legend=alt.Legend(orient="left", title=None)),
        order=alt.Order('tier', sort='descending'),
        tooltip=['tier:N', 'avg_cpu_usage:Q']
  )
cpu.encoding.x.title = "Hour"
cpu.encoding.y.title = "Average Utilization"


# Power Utilization graph
pu = (
    alt.Chart()
    .transform_fold(['avg_measured_power_util', 'avg_production_power_util'])
    .encode(
        alt.X(
            'hour_index:N',
            axis=alt.Axis(labels=False, domain=False, ticks=False),
        ),
        alt.Y('value:Q'),
        color=alt.Color('key:N', legend=None),
        strokeDash=alt.StrokeDash('key:N', legend=None),
        tooltip=['hour_index:N', 'key:N', 'value:Q']
    )
    .mark_line().properties(title=alt.datum.cell + ': ' + alt.datum.cluster)
)


alt.layer(cpu, pu, data=hourly_usage).properties(
    width=1200,
    height=300,
    title="Average CPU and Power Utilization").configure_axis(grid=False)

alt.LayerChart(...)

We can adapt the previous queries to calculate the average CPU and power utilizations per day by tier (i.e. cappable workloads or production) for each PDU in cell `b`.

In [ ]:
%%bigquery cluster_and_power_data_df
WITH
  machines_in_cell AS (
    SELECT machine_id, pdu, cell
    FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
    WHERE cell = 'b'
  ),
  cpu_capacities AS (
    SELECT pdu, cell, SUM(cpu_cap) AS cpu_capacity
    FROM
      (
        SELECT machine_id, MAX(capacity.cpus) AS cpu_cap,
        FROM `google.com:google-cluster-data`.clusterdata_2019_b.machine_events
        GROUP BY 1
      )
    JOIN machines_in_cell
      USING (machine_id)
    GROUP BY 1, 2
  ),
  per_instance_usage_priority AS (
    SELECT
      u.time AS time,
      u.collection_id AS collection_id,
      u.instance_index AS instance_index,
      e.priority AS priority,
      IF(e.priority < 120, 'cappable', 'production') AS tier,
      u.cpu_usage AS cpu_usage,
      m.pdu
    FROM
      (
        SELECT
          start_time AS time,
          collection_id,
          instance_index,
          machine_id,
          average_usage.cpus AS cpu_usage
        FROM `google.com:google-cluster-data`.clusterdata_2019_b.instance_usage
        WHERE
          (alloc_collection_id IS NULL OR alloc_collection_id = 0)
          AND (end_time - start_time) >= (5 * 60 * 1e6)
      ) AS u
    JOIN
      (
        SELECT collection_id, instance_index, machine_id, MAX(priority) AS priority
        FROM `google.com:google-cluster-data`.clusterdata_2019_b.instance_events
        WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        GROUP BY 1, 2, 3
      ) AS e
      ON
        u.collection_id = e.collection_id
        AND u.instance_index = e.instance_index
        AND u.machine_id = e.machine_id
    JOIN machines_in_cell AS m
      ON m.machine_id = u.machine_id
  )
SELECT *
FROM
  (
    SELECT
      CAST(FLOOR(time / (1e6 * 60 * 60 * 24)) AS INT64) AS day_index,
      pdu,
      tier,
      SUM(cpu_usage) / (12 * 24 * ANY_VALUE(cpu_capacity)) AS avg_cpu_usage,
    FROM per_instance_usage_priority
    JOIN cpu_capacities
      USING (pdu)
    GROUP BY 1, 2, 3
  )
JOIN
  (
    SELECT
      CAST(FLOOR((time - 6e8 + 3e8) / (1e6 * 60 * 60 * 24)) AS INT64) AS day_index,
      pdu,
      AVG(measured_power_util) AS avg_measured_power_util,
      AVG(production_power_util) AS avg_production_power_util
    FROM `google.com:google-cluster-data`.`powerdata_2019.cellb_pdu*`
    GROUP BY 1, 2
  )
  USING (pdu, day_index)

Executing query with job ID: b6ed98c0-5ed9-42ea-93b3-8a145fa0491a
Query executing: 0.58s


ERROR:
 403 GET https://bigquery.googleapis.com/bigquery/v2/projects/black-backup-489114-h7/queries/b6ed98c0-5ed9-42ea-93b3-8a145fa0491a?maxResults=0&location=US&prettyPrint=false: Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas

Location: US
Job ID: b6ed98c0-5ed9-42ea-93b3-8a145fa0491a



In [ ]:
cluster_and_power_data_df.describe()

AttributeError: 'QueryJob' object has no attribute 'describe'

### Be careful joining on `time` directly!
Note that in the queries above we converted `time` from the cluster and power datasets to an `hour_index` which we then use to join the two datasets. We don't want to join on `time` directly due to the how the datasets are structured.

The power trace `time` values are each aligned to the 5-minute mark. For example, there's a `time` at 600s, 900s, 1200s but never 700s or 601s. The cluster trace `time` values have no specific alignment. If we were to join the two datasets on `time`, we'd end up dropping a lot of data!

You may have also noticed that there are `day_index` has 32 unique values in the query above, despite May having 31 days. This is because timestamps in the data sets are represented as microseconds since 600 seconsd before the start of the trace period, May 01 2019 at 00:00 PT.

## Recreating Graphs from the Paper

Below, the power traces are used to re-create figures from [the paper](https://research.google/pubs/pub49032/).

In [ ]:
def histogram_data(filter='pdu%', agg_by='pdu', util_type='measured_power_util'):
  query = """
  SELECT bin as bins, SUM(count) as counts
  FROM (
    SELECT {agg_by},
           ROUND(CAST({util_type} / 0.0001 as INT64) * 0.0001, 3) as bin,
           COUNT(*) as count
    FROM (
      SELECT {agg_by},
             time,
             {util_type},
      FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
      WHERE NOT bad_measurement_data
            AND NOT bad_production_power_data
            AND {agg_by} LIKE '{filter}'
            AND NOT cell in ('i', 'j')
    )
    GROUP BY 1, 2
  ) GROUP BY 1 ORDER BY 1;
  """.format(**{'filter': filter, 'agg_by': agg_by, 'util_type': util_type})
  return client.query(query).to_dataframe()

In [ ]:
pdu_df = histogram_data()
cluster_df = histogram_data('%', 'cell')
prod_pdu_df = histogram_data(util_type='production_power_util')
prod_cluster_df = histogram_data('%', 'cell', util_type='production_power_util')

In [ ]:
def make_cdf(p_df, c_df, title):
  pdu_counts, pdu_bins = p_df.counts, p_df.bins
  cluster_counts, cluster_bins = c_df.counts, c_df.bins

  pdu_cdf = np.cumsum (list(pdu_counts))
  pdu_cdf = (1.0 * pdu_cdf) / pdu_cdf[-1]
  cluster_cdf = np.cumsum (list(cluster_counts))
  cluster_cdf = (1.0 * cluster_cdf) / cluster_cdf[-1]

  pdu_cdf_graph = alt.Chart(pd.DataFrame(
      {'bins': pdu_bins, 'cdf': pdu_cdf})).mark_point(size=.1).encode(
        x=alt.X('bins', scale=alt.Scale(domain=[0.4, 0.90])),
        y=alt.Y('cdf'),
        color=alt.value('steelblue')
  )

  cluster_cdf_graph = alt.Chart(pd.DataFrame(
      {'bins': cluster_bins, 'cdf': cluster_cdf})).mark_point(size=.1).encode(
        x=alt.X('bins', scale=alt.Scale(domain=[0.4, 0.90])),
        y=alt.Y('cdf'),
        color=alt.value('forestgreen')
  )

  return (pdu_cdf_graph + cluster_cdf_graph).properties(title=title)


alt.hconcat(make_cdf(pdu_df, cluster_df, "Measured Power Util"), make_cdf(
    prod_pdu_df, prod_cluster_df, "Production Power Util"))

end of document - what follows is my analysis...

In [ ]:
%%bigquery machines_per_pdu_df
SELECT
  cell,
  pdu,
  COUNT(DISTINCT machine_id) AS machines
FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
GROUP BY cell, pdu
ORDER BY machines DESC;


In [ ]:
print(machines_per_pdu_df.to_string())

In [ ]:
%%bigquery pdu_machine_counts_cell_a_df
SELECT
  pdu,
  COUNT(DISTINCT machine_id) AS machine_count
FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
WHERE cell = 'a'
GROUP BY pdu
ORDER BY pdu;


In [ ]:
print(pdu_machine_counts_cell_a_df.to_string())

**Average CPU Utilisation**

In [ ]:
%%bigquery mean_cpu_cell_a_df
WITH
  machines_in_cell AS (
    SELECT machine_id, pdu
    FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
    WHERE cell = 'a'
  ),

  cpu_capacities AS (
    SELECT
      m.pdu,
      SUM(cpu_cap) AS cpu_capacity
    FROM (
      SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
      FROM `google.com:google-cluster-data`.clusterdata_2019_a.machine_events
      GROUP BY machine_id
    ) caps
    JOIN machines_in_cell m
      ON caps.machine_id = m.machine_id
    GROUP BY m.pdu
  ),

  per_instance_usage AS (
    SELECT
      start_time AS time,
      machine_id,
      average_usage.cpus AS cpu_usage
    FROM `google.com:google-cluster-data`.clusterdata_2019_a.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
      AND (end_time - start_time) >= (5 * 60 * 1e6)
  ),

  cpu_by_pdu_day AS (
    SELECT
      CAST(FLOOR(time / (1e6 * 60 * 60 * 24)) AS INT64) AS day_index,
      m.pdu,
      SUM(u.cpu_usage) AS total_cpu_used
    FROM per_instance_usage u
    JOIN machines_in_cell m
      ON u.machine_id = m.machine_id
    GROUP BY day_index, m.pdu
  )

--SELECT
--  AVG(total_cpu_used / (12 * 24 * cpu_capacity)) AS mean_cpu_util_cell_a
--FROM cpu_by_pdu_day
--JOIN cpu_capacities
--  USING (pdu);

  SELECT
  AVG(cell_cpu_util) AS mean_cpu_util_cell_a
FROM (
  SELECT
    day_index,
    SUM(total_cpu_used) /
      (12 * 24 * SUM(cpu_capacity)) AS cell_cpu_util
  FROM cpu_by_pdu_day
  JOIN cpu_capacities
    USING (pdu)
  GROUP BY day_index
);


Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
print(mean_cpu_cell_a_df.to_string())

   mean_cpu_util_cell_a
0              0.543344


**AVERAGE CPU UTILISATION RESULTS**...

cell A = 0.54 (54%)

cell B = 0.52 (52%)


In [ ]:
%%bigquery mean_power_cell_a_df
WITH daily_pdu_power AS (
  SELECT
    CAST(FLOOR((time - 6e8 + 3e8) / (1e6 * 60 * 60 * 24)) AS INT64) AS day_index,
    pdu,
    AVG(measured_power_util) AS avg_measured_power_util,
    AVG(production_power_util) AS avg_production_power_util
  FROM `google.com:google-cluster-data`.`powerdata_2019.cella_pdu*`
  GROUP BY day_index, pdu
)

SELECT
  AVG(avg_measured_power_util) AS mean_measured_power_util_cell_a,
  AVG(avg_production_power_util) AS mean_production_power_util_cell_a
FROM daily_pdu_power;


Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
print(mean_power_cell_a_df.to_string())

   mean_measured_power_util_cell_a  mean_production_power_util_cell_a
0                         0.718469                           0.607545


In [ ]:
%%bigquery machines_per_pdu_cell_a_df
SELECT
  pdu,
  COUNT(DISTINCT machine_id) AS machine_count
FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
WHERE cell = 'a'
GROUP BY pdu
ORDER BY pdu;


Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
machines_per_pdu_cell_a_df['machine_count'].mean()


np.float64(2000.4)

**drawing a line under the above analysis!**

let's do some basic querying to understand the shape of the data and the google infra in question...

In [ ]:
%%bigquery machines_per_pdu_df
SELECT
  cell,
  pdu,
  COUNT(DISTINCT machine_id) AS machines
FROM `google.com:google-cluster-data`.powerdata_2019.machine_to_pdu_mapping
GROUP BY cell, pdu
ORDER BY cell ASC;

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
print(machines_per_pdu_df.to_string())

   cell    pdu  machines
0     a   pdu6      1738
1     a   pdu7      1534
2     a   pdu8      2010
3     a   pdu9      2478
4     a  pdu10      2242
5     b  pdu11      1932
6     b  pdu15      2225
7     b  pdu14      1851
8     b  pdu13      1904
9     b  pdu12      2138
10    c  pdu38      2380
11    c  pdu41      2156
12    c  pdu42      2370
13    c  pdu43      1901
14    c  pdu39      2279
15    c  pdu40      2184
16    d  pdu37      2400
17    d  pdu36      2462
18    d  pdu35      2115
19    d  pdu34      1859
20    d  pdu33      1834
21    d  pdu32      1906
22    e  pdu29      2121
23    e  pdu31      2457
24    e  pdu30      2641
25    e  pdu26      2500
26    e  pdu27      2232
27    e  pdu28      2172
28    f  pdu25      1265
29    f  pdu24      1245
30    f  pdu23      1014
31    f  pdu22      1124
32    f  pdu21      1026
33    f  pdu19      1163
34    f  pdu18      1392
35    f  pdu17      1295
36    f  pdu16      1534
37    f  pdu20      1145
38    g   pdu1      2550


In [ ]:
%%bigquery
SELECT server_count
FROM (
  SELECT COUNT(DISTINCT machine_id) AS server_count
  FROM `google.com:google-cluster-data`.clusterdata_2019_a.machine_events
)

Query is running:   0%|          |

Downloading:   0%|          |

,server_count
0,10001


so, we've shown that cell A and B both have 5 PDUs and both have ~10k machines.
drawing another line under this...

In [ ]:
%%bigquery
SELECT
  average_power_util_10,
  average_power_util_9,
  average_power_util_8,
  average_power_util_7,
  average_power_util_6,
  (average_power_util_10 + average_power_util_9 + average_power_util_8 + average_power_util_7 + average_power_util_6) / 5 AS overall_average
FROM (
SELECT
  (SELECT AVG(measured_power_util)
   FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu10) AS average_power_util_10,
  (SELECT AVG(measured_power_util)
   FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu9) AS average_power_util_9,
   (SELECT AVG(measured_power_util)
   FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu8) AS average_power_util_8,
   (SELECT AVG(measured_power_util)
   FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu7) AS average_power_util_7,
   (SELECT AVG(measured_power_util)
   FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu6) AS average_power_util_6
)

Query is running:   0%|          |

Downloading:   0%|          |

,average_power_util_10,average_power_util_9,average_power_util_8,average_power_util_7,average_power_util_6,overall_average
0,0.735991,0.731638,0.706376,0.702223,0.715348,0.718315


so, average power utilisation for cell A = 0.718315

**watch out** for 'measured' (measured_power_util) vs 'production' (production_power_util) power utilisation!

production is where its attributable to production workloads.



In [ ]:
%%bigquery
WITH daily_pdu_power AS (
  SELECT
    pdu,
    AVG(measured_power_util) AS avg_measured_power_util
  FROM `google.com:google-cluster-data.powerdata_2019.cella_pdu*`
  GROUP BY pdu
)
SELECT
  AVG(avg_measured_power_util) AS mean_measured_power_util
FROM daily_pdu_power

In [ ]:
%%bigquery power_cell_a_df

WITH power_observations AS (

  SELECT
    'pdu6' AS pdu,
    measured_power_util,
    bad_measurement_data
  FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu6

  UNION ALL

  SELECT
    'pdu7' AS pdu,
    measured_power_util,
    bad_measurement_data
  FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu7

  UNION ALL

  SELECT
    'pdu8' AS pdu,
    measured_power_util,
    bad_measurement_data
  FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu8

  UNION ALL

  SELECT
    'pdu9' AS pdu,
    measured_power_util,
    bad_measurement_data
  FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu9

  UNION ALL

  SELECT
    'pdu10' AS pdu,
    measured_power_util,
    bad_measurement_data
  FROM `google.com:google-cluster-data`.powerdata_2019.cella_pdu10
)

SELECT
  pdu,
  AVG(measured_power_util) AS mean_measured_power_util,
  COUNT(*) AS observation_count
FROM power_observations
WHERE bad_measurement_data = FALSE
GROUP BY pdu
ORDER BY pdu;

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
power_cell_a_df

,pdu,mean_measured_power_util,observation_count
0,pdu10,0.735991,8928
1,pdu6,0.715348,8928
2,pdu7,0.702223,8928
3,pdu8,0.706376,8928
4,pdu9,0.731638,8928


In [ ]:
power_cell_a_df["mean_measured_power_util"].mean()

np.float64(0.7183154121863805)

In [ ]:
power_cell_a_df["observation_count"]

,observation_count
0,8928
1,8928
2,8928
3,8928
4,8928
